# 📘 Chapter 6 — Decision Trees

## Topic 1: What Is a Decision Tree?

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Beginning of the chapter — **Introduction to Decision Trees**
* Section: **Training and Visualizing a Decision Tree**
* First example using the **Iris dataset**

---

## 🧠 1. Core Idea of Decision Trees

### 📌 Definition

A Decision Tree is a supervised learning algorithm that:
* Splits the dataset into smaller subsets
* Based on **feature thresholds**
* Forms a **tree-like structure** of decisions

It can be used for **classification** and **regression**.

---

## 🌳 2. How It Works (Conceptually)

Imagine asking a sequence of yes/no questions:
```
Is petal length < 2.5?
   ├── Yes → Class A
   └── No:
         Is petal width < 1.8?
              ├── Yes → Class B
              └── No → Class C
```

* Each **internal node** → tests a feature, splits the dataset
* Each **leaf node** → outputs a prediction

---

## 🎯 3. Why Decision Trees Are Powerful

**✅ Advantages:**
* Very interpretable — you can read the logic directly
* **No feature scaling required**
* Works with nonlinear data
* Handles numerical + categorical features (in theory)

**❌ Weaknesses:**
* Prone to **overfitting**
* **High variance** — small data changes can produce very different trees

---

## 📊 4. Geometric Intuition

Decision Trees create:
* **Axis-aligned splits** — always parallel to a feature axis
* Partition the feature space into **rectangles**

They do **NOT** create:
* Diagonal boundaries
* Smooth curves

> 📝 **Key idea:** Decision Trees produce "boxy" decision regions. This is a fundamental limitation of the algorithm — complex diagonal boundaries require many splits to approximate.

---

## 💻 Code: Training a Simple Decision Tree (Iris Example)


In [1]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text

# Load dataset
iris = load_iris()
X = iris.data[:, 2:]  # petal length & width
y = iris.target

# Train tree
tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf.fit(X, y)

# Print tree
print(export_text(tree_clf, feature_names=["petal_length", "petal_width"]))

|--- petal_length <= 2.45
|   |--- class: 0
|--- petal_length >  2.45
|   |--- petal_width <= 1.75
|   |   |--- class: 1
|   |--- petal_width >  1.75
|   |   |--- class: 2



## 🔑 Key Parameter: `max_depth`
```python
max_depth=2
```

We **limit depth** to prevent overfitting — we will study this deeply later.

---

## 🧠 What You Should Notice

* The model only uses **petal length & width** — two features are enough
* It finds **thresholds automatically** (no manual rules needed)
* It **recursively splits** data at each internal node
* It **stops when `max_depth` is reached** — this is the main complexity control

---

## 📝 Key Takeaways — Topic 1

| Concept | Core idea |
|---|---|
| Decision Tree | Sequence of yes/no questions on feature thresholds |
| Internal node | Tests a feature → splits the data |
| Leaf node | Outputs a prediction (class or value) |
| Decision regions | Axis-aligned rectangles ("boxy") |
| `max_depth` | Primary knob to control overfitting |
| Scaling | **Not required** — trees use thresholds, not distances |

# 📘 Topic 2 — How Does the Tree Decide Where to Split?

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section: **The CART Training Algorithm**
* Subsection: **Gini Impurity**
* The formula for Gini

---

## 🧠 Core Question

How does the tree decide:
* **Which feature** to split on?
* **At which threshold?**

> **Answer:** It searches for the split that **minimizes impurity**.

---

## 📊 1. Gini Impurity

For classification, the default metric is:

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

Where:
* $p_k$ = proportion of class $k$ in the node
* $K$ = number of classes

---

## 🎯 Intuition

| Node composition | Gini | Meaning |
|---|---|---|
| 100% one class | $0$ | Perfectly pure ✅ |
| Mixed evenly | High | Very impure ❌ |
| Binary, 50/50 split | $0.5$ | Maximum impurity |

**Example — impure node (50/50):**

$$G = 1 - (0.5^2 + 0.5^2) = 1 - (0.25 + 0.25) = 0.5$$

**Example — pure node (100% class A):**

$$G = 1 - (1^2) = 0$$

---

## 🧠 What the Tree Actually Does

For **every feature**, for **every possible threshold**:

1. Compute the **weighted average impurity** of left and right child:

$$J = \frac{m_{left}}{m} G_{left} + \frac{m_{right}}{m} G_{right}$$

2. Pick the split that **minimizes $J$**

> 📝 **Key idea:** This is **greedy optimization** — the tree finds the locally best split at each node, not the globally optimal tree. This is fast but can lead to suboptimal overall structure.

---

## 📝 Key Takeaways — Topic 2

| Concept | Core idea |
|---|---|
| Gini impurity | Measures how mixed classes are in a node |
| Pure node | Gini = 0 — all one class |
| Max impurity (binary) | Gini = 0.5 — perfectly mixed |
| Split criterion | Minimize **weighted** Gini of children |
| Search strategy | Try every feature × every threshold → greedy |

## ✅ Gini Impurity Check — Node: 80% A / 20% B

$$G = 1 - (0.8^2 + 0.2^2) = 1 - (0.64 + 0.04) = 1 - 0.68 = 0.32$$

### 🎯 Interpretation

A Gini of **0.32** means: the node is **somewhat pure**, but not perfectly pure — some class mixing remains.

| Class Distribution | Gini |
|---|---|
| 100% / 0% | $0$ |
| 80% / 20% | $0.32$ |
| 50% / 50% | $0.5$ |

> Impurity increases as class balance becomes more even.

---

# 📘 Topic 3 — The CART Algorithm (Greedy Splitting)

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section: **The CART Training Algorithm**
* The equation showing weighted impurity minimization

---

## 🧠 1. What Is CART?

**CART = Classification And Regression Tree**

| Fact | Detail |
|---|---|
| Tree structure | Always **binary** (exactly two children per split) |
| Search strategy | **Greedy** — locally optimal at each node |
| Look-ahead | ❌ None — does not reconsider earlier splits |
| Global optimality | ❌ Not guaranteed |

---

## 📊 2. What Does CART Minimize?

For each possible split $(k, t_k)$ — feature $k$ at threshold $t_k$:

$$J(k, t_k) = \frac{m_{left}}{m} G_{left} + \frac{m_{right}}{m} G_{right}$$

Where:
* $m$ = total samples at node
* $m_{left}$, $m_{right}$ = samples in left/right child
* $G_{left}$, $G_{right}$ = Gini impurity of each child

---

## 🎯 Key Idea

> The tree chooses the split that minimizes the **weighted impurity of the children** — not the parent impurity.

---

## 🧠 Intuition

The tree asks at every node:
> *"If I split here, how pure do my children become?"*

It does this for **every feature** × **every threshold**, then picks the best one.

---

## 💡 Why Is It Greedy?

| Property | Meaning |
|---|---|
| No look-ahead | Picks best **immediate** reduction only |
| No backtracking | Does not reconsider earlier splits |
| No combinations | Does not explore multi-split interactions |

> 📝 **Key idea:** Finding the globally optimal tree is an **NP-complete problem** — so CART trades global optimality for speed by being greedy. This is why pruning and `max_depth` matter so much.

## ✅ Weighted Impurity — Why It Matters

**Without weighting** — the algorithm could prefer splits that create one tiny pure node and one huge impure node, and incorrectly think it's a good split.

### 🎯 Concrete Example

* Left child → **1 sample** → perfectly pure → Gini = 0
* Right child → **999 samples** → very impure → Gini = 0.5

**Without weighting:**
$$G_{left} + G_{right} = 0 + 0.5 = 0.5 \quad \text{(looks acceptable)}$$

**With weighting:**
$$J = \frac{1}{1000}(0) + \frac{999}{1000}(0.5) \approx 0.4995 \quad \text{(reveals it's terrible)}$$

> 📝 **Core insight:** Weighting ensures **large nodes influence the decision more than tiny nodes**. Without it, the tree would over-prefer tiny pure splits that leave most data untouched.

---

# 📘 Topic 4 — Decision Tree Overfitting

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section: **Regularization Hyperparameters**
* Keywords: `max_depth`, `min_samples_split`, `min_samples_leaf`

---

## 🧠 Why Trees Overfit Easily

Decision Trees, if unconstrained:
* Keep splitting until **all nodes are pure**
* Can **memorize** training data
* Have **high variance** — small data changes → very different trees
* Create extremely complex "boxy" partitions

---

## 📊 What Happens When a Tree Is Too Deep?

* Creates **tiny regions** around individual training points
* Captures **noise** as if it were signal
* Training error → near **zero**
* Test error → **high**

> This is classic **high variance / overfitting**.

---

## 🎯 How We Control Overfitting

| Hyperparameter | Effect |
|---|---|
| `max_depth` | Limits tree height — primary complexity control |
| `min_samples_split` | Min samples required to split a node |
| `min_samples_leaf` | Min samples required in a leaf node |
| `max_leaf_nodes` | Max number of leaves allowed |

All of these **force smoother, less complex decision boundaries**.

---

## 💻 Code Example
```python
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
tree_clf.fit(X_train, y_train)
```

---

## 📝 Key Takeaways — Topic 4

| Concept | Core idea |
|---|---|
| Unconstrained tree | Memorizes training data → high variance |
| `max_depth` | Hardest stop — cuts the tree at a fixed level |
| `min_samples_leaf` | Prevents splits that create tiny, noisy leaves |
| Regularization goal | Trade some bias for lower variance → better generalization |

> 📝 **Key idea:** Every regularization parameter above **increases bias slightly** in exchange for **reducing variance** — the same bias–variance tradeoff from Chapters 4 and 5, now applied to tree structure.

## ✅ Effect of Increasing `max_depth` (e.g., 3 → 50)

| Metric | Direction | Why |
|---|---|---|
| Training error | ⬇ Decreases | Tree memorizes training data |
| Variance | ⬆ Increases | Small data changes → very different trees |
| Bias | ⬇ Decreases | More flexibility → captures complex patterns |

> 📝 **Key idea:** Increasing depth = ↓ Bias + ↑ Variance + ↓ Training error = **overfitting risk**. This is the bias–variance tradeoff applied to tree structure.

---

# 📘 Topic 5 — Decision Tree Regression

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section: **Regression**
* Example with **1D dataset**
* The figure showing **piecewise constant prediction**

---

## 🧠 How Regression Trees Differ

| | Classification Tree | Regression Tree |
|---|---|---|
| Predicts | Class label | Numerical value |
| Split criterion | Gini / Entropy | **MSE** |
| Leaf output | Majority class | **Mean of samples in leaf** |

---

## 📊 What Does a Regression Tree Do?

1. Splits the feature space (same greedy CART algorithm)
2. Each leaf predicts the **mean value** of training samples in that leaf
3. Creates a **piecewise constant function** — prediction is flat within each region

---

## 🎯 Key Insight

| Tree Type | Optimization goal |
|---|---|
| Classification | Minimize **weighted Gini impurity** of children |
| Regression | Minimize **weighted MSE** of children |

$$J(k, t_k) = \frac{m_{left}}{m} \text{MSE}_{left} + \frac{m_{right}}{m} \text{MSE}_{right}$$

> 📝 **Key idea:** A regression tree produces a **staircase-shaped** prediction curve — not smooth. Deeper trees → more steps → tighter fit to training data → same overfitting risk as classification trees.

In [2]:
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# Sample data
X = np.random.rand(100, 1)
y = 4 * (X[:, 0] - 0.5) ** 2 + np.random.randn(100) / 10

# Train regressor
tree_reg = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg.fit(X, y)

DecisionTreeRegressor(max_depth=2, random_state=42)

## ✅ Regression Tree Leaf — What's Stored

The leaf stores the **mean of all training target values** that fell into that region.

### 🎯 Example

Leaf contains: `[2, 4, 6, 8]`

$$\text{Prediction} = \frac{2 + 4 + 6 + 8}{4} = 5$$

Every new sample that lands in that region → prediction = **5**

### 🧠 Why Mean?

> The mean **minimizes squared error** — which is exactly why MSE is used as the split criterion. The leaf prediction and the split criterion are mathematically consistent.

---

# 📘 Topic 6 — Decision Tree Instability (High Variance)

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section discussing **instability**
* The part mentioning **rotation sensitivity**

---

## 🧠 Core Idea

Decision Trees are **highly unstable** because:
* Small changes in training data
* → Can produce very different splits
* → Which change the **entire tree structure** downstream

They are sensitive to:
* **Noise** — one noisy point can redirect a split
* **Outliers** — extreme values can dominate a threshold
* **Feature rotation** — axis-aligned splits can't adapt

---

## 🎯 Critical Concept: Axis-Aligned Splits

Decision Trees **only split along feature axes** — every boundary is parallel to a feature axis.
```
✅ Can do:          ❌ Cannot do naturally:
|                   /
|___                /
    |              /
```

**If you rotate the dataset** → the tree structure changes completely, because it must now approximate a diagonal boundary with many axis-aligned steps.

---

## 🧠 Why This Matters (Bridge to Chapter 7)

| Problem | Cause | Fix |
|---|---|---|
| High variance | Small data changes → different trees | **Random Forests** (average many trees) |
| Rotation sensitivity | Axis-aligned splits only | **Random Forests** (random feature subsets) |
| Overfitting | Greedy splits memorize noise | `max_depth`, `min_samples_leaf` |

> 📝 **Key idea:** A single Decision Tree is a **weak, unstable learner**. This instability is actually *exploited* by Random Forests — by averaging many different trees trained on slightly different data, variance cancels out while bias stays low. This is the core idea of Chapter 7.

## ✅ Why Rotation Drastic Changes a Decision Tree

Decision Trees split only on:
```
feature_j < threshold
```

Splits are always **vertical or horizontal** — never diagonal.

### 🎯 What Happens When You Rotate the Dataset?

* New features become **linear combinations** of original features
* What was a clean vertical split → now requires a **diagonal boundary**
* Trees **cannot split diagonally** → must approximate with many small axis-aligned steps
* This changes: which feature is chosen, where splits occur, entire tree structure

> 📝 **One sentence to remember:** Because Decision Trees can only create axis-aligned splits, rotating the feature space changes the geometry of optimal splits, **forcing the tree to restructure completely** — not because weights change, but because of the geometric constraints of the model.

---

# 📘 Topic 7 — Computational Complexity of Training

---

## A) Where to Read

In `6_Decision Trees.pdf`:
* Section mentioning **training complexity**
* **Big-O notation** discussion

---

## 🧠 Training Complexity

At each node, the algorithm:
* Tests **all $n$ features**
* For each feature → tries all possible **thresholds**
* Across all samples at that node

For a dataset with $m$ samples and $n$ features:

$$\text{Training complexity} \approx O(n \cdot m \log m)$$

---

## 🎯 Why $\log m$?

| Reason | Detail |
|---|---|
| Balanced tree depth | $\approx \log_2(m)$ levels |
| Each split | Roughly halves the data |
| Work per level | $O(n \cdot m)$ — scan all features × samples |
| Total | $O(n \cdot m) \times O(\log m) = O(n \cdot m \log m)$ |

---

## 📌 Prediction Complexity

Once trained, prediction is very fast:

$$\text{Prediction complexity} = O(\log m)$$

Just traverse the tree from root to leaf — one comparison per level.

---

## 📝 Key Takeaways — Topic 7

| Operation | Complexity | Why |
|---|---|---|
| Training | $O(n \cdot m \log m)$ | All features × thresholds × tree depth |
| Prediction | $O(\log m)$ | One path from root to leaf |

> 📝 **Key idea:** Training is moderately expensive (scales with $m \log m$), but prediction is **extremely fast** — just $\log m$ comparisons. This makes Decision Trees attractive for real-time inference once trained.

## ✅ Why Prediction Is Much Faster Than Training

### During Training:
The algorithm must:
* Try **all features** × **all possible thresholds**
* Compute **impurity** for every candidate split
* Repeat at **every node** across the entire tree

$$\text{Complexity: } O(n \cdot m \log m)$$

### During Prediction:
We only:
* Start at the root
* Check **one condition** per node
* Move left or right
* Repeat until reaching a leaf

**No searching. No optimization. No impurity calculation. Just simple comparisons.**

$$\text{Complexity: } O(\log m)$$

> 📝 **Key insight:** Training **explores** many possible trees. Prediction **follows** one fixed path. That's why prediction is dramatically cheaper.

---

# 📘 Chapter 6 — Full Recap

---

| Topic | Core Idea |
|---|---|
| **1. What is a Decision Tree** | Sequence of yes/no questions on feature thresholds → class or value |
| **2. Geometric interpretation** | Axis-aligned splits → "boxy" rectangular decision regions |
| **3. Gini impurity** | $G = 1 - \sum p_k^2$ — measures class mixing; 0 = pure, 0.5 = max impurity (binary) |
| **4. CART algorithm** | Greedy search: minimize weighted Gini of children at every node |
| **5. Overfitting & regularization** | Unconstrained trees memorize noise; control with `max_depth`, `min_samples_leaf` |
| **6. Regression trees** | Same CART idea; leaves store **mean** of targets; minimizes MSE |
| **7. Instability** | Axis-aligned splits → high variance; rotation changes tree completely |
| **8. Computational complexity** | Training: $O(n \cdot m \log m)$; Prediction: $O(\log m)$ |

---

## 🧠 Unifying Thread

$$\underbrace{\text{Greedy splits}}_{\text{CART}} \rightarrow \underbrace{\text{Axis-aligned regions}}_{\text{geometry}} \rightarrow \underbrace{\text{High variance}}_{\text{instability}} \rightarrow \underbrace{\text{Random Forests}}_{\text{Chapter 7 fix}}$$

> 📝 **Chapter 6 in one sentence:** Decision Trees are powerful, interpretable, and fast to predict — but their greedy, axis-aligned, high-variance nature makes them weak learners that become truly powerful only when **combined into ensembles** (Chapter 7).

# 🌳 Chapter 6 — Decision Trees: Deep Mastery Summary

---

## 1️⃣ What Is a Decision Tree?

A Decision Tree is a supervised learning algorithm that:
* Recursively **splits** the dataset based on feature thresholds
* Forms a **tree structure** of decisions
* Makes predictions at **leaf nodes**

Used for **classification** and **regression**.

---

## 2️⃣ Structure of a Tree

| Component | Role |
|---|---|
| **Internal node** | Tests a feature: `feature_j < threshold` |
| **Branch** | Outcome of the test (True / False) |
| **Leaf node** | Stores the prediction |

---

## 3️⃣ Geometric Interpretation

Decision Trees create:
* **Axis-aligned splits** — vertical or horizontal boundaries only
* **Rectangular decision regions**

They **cannot** create diagonal splits.

> **Key implication:** Rotating the dataset changes the tree structure drastically — the model must approximate diagonal boundaries with many axis-aligned steps.

---

## 4️⃣ How Does the Tree Choose a Split? (CART)

**CART = Classification And Regression Trees**

| Property | Detail |
|---|---|
| Split type | **Binary only** |
| Search strategy | **Greedy** — locally optimal at each node |
| Look-ahead | ❌ None |
| Global optimality | ❌ Not guaranteed |

---

## 5️⃣ Classification Trees

### Impurity Measure: Gini Impurity

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

| Node type | Gini |
|---|---|
| Perfectly pure | $0$ |
| 50/50 binary split | $0.5$ |
| Mixed but skewed | Between $0$ and $0.5$ |

### Split Objective

$$J = \frac{m_{left}}{m} G_{left} + \frac{m_{right}}{m} G_{right}$$

> Minimizes **weighted child impurity** — not parent impurity. Weighting prevents preference for tiny pure nodes over large impure ones.

---

## 6️⃣ Regression Trees

* Split criterion: **Minimize weighted MSE** of children
* Leaf prediction: **Mean of target values** in that leaf (mean minimizes squared error)
* Produces a **piecewise constant** (step-like) prediction curve

---

## 7️⃣ Overfitting & Regularization

| Hyperparameter | Effect |
|---|---|
| `max_depth` | Limits tree height |
| `min_samples_split` | Minimum samples required to split |
| `min_samples_leaf` | Minimum samples per leaf |
| `max_leaf_nodes` | Limits number of leaves |

### Bias–Variance Tradeoff

| Depth | Bias | Variance | Risk |
|---|---|---|---|
| Shallow | High | Low | Underfit |
| Deep | Low | High | **Overfit** |

---

## 8️⃣ Instability

Decision Trees are sensitive to:
* **Small data changes** → different splits → different tree
* **Outliers** → can redirect splits
* **Feature rotation** → axis-aligned constraint forces complete restructure

> **Root cause:** The axis-aligned splitting constraint makes the geometry fragile.

---

## 9️⃣ Computational Complexity

| Operation | Complexity | Why |
|---|---|---|
| Training | $O(n \cdot m \log m)$ | All features × thresholds × tree depth |
| Prediction | $O(\log m)$ | One path from root to leaf |

---

## 🔟 Strengths & Weaknesses

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Interpretable | High variance |
| No feature scaling required | Overfits easily |
| Handles nonlinear patterns | Axis-aligned limitation |
| Works with numerical & categorical | Unstable |

---

## 🧠 Conceptual Core — If You Remember Only 5 Things

1. Trees split feature space into **rectangles**
2. Splits chosen by minimizing **weighted impurity**
3. Leaves store **majority class** (classification) or **mean** (regression)
4. Deep trees = **low bias, high variance**
5. Trees are **unstable** because of axis-aligned splits

---

## 🎓 Mastery Status

| Area | ✅ |
|---|---|
| Geometry | ✅ |
| Math (Gini + MSE) | ✅ |
| Optimization objective | ✅ |
| Bias–variance behavior | ✅ |
| Instability | ✅ |
| Computational complexity | ✅ |

$$\underbrace{\text{Greedy splits}}_{\text{CART}} \rightarrow \underbrace{\text{Axis-aligned regions}}_{\text{geometry}} \rightarrow \underbrace{\text{High variance}}_{\text{instability}} \rightarrow \underbrace{\text{Random Forests}}_{\text{Chapter 7 fix}}$$

# 📘 Chapter 6 — Decision Trees: Exercises Q&A

---

**Q1. What is the approximate depth of a Decision Tree trained (without restrictions) on a training set with 1 million instances?**

A fully grown (balanced) Decision Tree has depth approximately $\log_2(m)$, where $m$ is the number of instances:

$$\log_2(1{,}000{,}000) \approx 20$$

> Approximate depth ≈ **20 levels**

---

**Q2. Is a node's Gini impurity generally lower or greater than its parent's? Always or generally?**

A child node's Gini impurity is **generally lower** than its parent's — but **not always**.

* The split guarantees that the **weighted average** of children's impurities is lower than the parent's
* But an **individual child** can have higher impurity than the parent (if the other child is much purer)

> Only the **weighted average** is guaranteed to decrease — not each child individually.

---

**Q3. If a Decision Tree is overfitting, is it a good idea to decrease `max_depth`?**

**Yes.** Decreasing `max_depth`:
* Restricts tree complexity
* Slightly increases bias
* Reduces variance
* Prevents memorizing noise

> This is the primary regularization lever for Decision Trees.

---

**Q4. If a Decision Tree is underfitting, is it a good idea to scale the input features?**

**No.** Decision Trees are **invariant to monotonic feature scaling** — they use thresholds, not distances or dot products.

> Underfitting should be addressed by **increasing model complexity** (e.g., increase `max_depth`, decrease `min_samples_leaf`).

---

**Q5. Training takes 1 hour on 1M instances. Roughly how long for 10M instances?**

Training complexity is $O(m \log m)$. The ratio:

$$\frac{10m \cdot \log(10m)}{m \cdot \log(m)} = 10 \cdot \frac{\log(10{,}000{,}000)}{\log(1{,}000{,}000)} = 10 \cdot \frac{\approx 23.25}{\approx 19.93} \approx 11.7$$

> Roughly **~11–12 hours** — slightly more than 10× due to the extra $\log$ factor.

---

**Q6. If your training set contains 100,000 instances, will `presort=True` speed up training?**

**No.** For large datasets like 100,000 instances, `presort=True` typically **slows down training** — the overhead of presorting outweighs any splitting efficiency gains.

> `presort=True` is only beneficial for **small datasets** where the sort overhead is negligible.

7. Train and fine-tune a Decision Tree for the moons dataset.

a. Generate a moons dataset using make_moons(n_samples=10000, noise=0.4).

b. Split it into a training set and a test set using train_test_split().

c. Use grid search with cross-validation (with the help of the GridSearchCV
class) to find good hyperparameter values for a DecisionTreeClassifier.
Hint: try various values for max_leaf_nodes.

d. Train it on the full training set using these hyperparameters, and measure
your model’s performance on the test set. You should get roughly 85% to 87%
accuracy

In [3]:
# Chapter 6, Exercise 7 — Decision Tree on the moons dataset
# a) make_moons(n_samples=10000, noise=0.4)
# b) train_test_split
# c) GridSearchCV over max_leaf_nodes (and a couple other helpful params)
# d) Train best model and evaluate on test set (aim ~85–87% accuracy)

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Reproducibility
RANDOM_STATE = 42

# a) Generate dataset
X, y = make_moons(n_samples=10_000, noise=0.4, random_state=RANDOM_STATE)

# b) Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# c) Grid search with CV
# Hint focus: max_leaf_nodes. We'll also try a couple related regularizers.
param_grid = {
    "max_leaf_nodes": list(range(2, 200)),           # main knob
    "min_samples_leaf": [1, 2, 5, 10, 20],           # helps with noisy moons
    "max_depth": [None, 2, 3, 4, 5, 6, 8, 10],       # optional but useful
}

base_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=base_tree,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1,
)

grid.fit(X_train, y_train)

print("Best CV accuracy:", grid.best_score_)
print("Best params:", grid.best_params_)

# d) Train best model on full training set and evaluate on test set
best_tree = grid.best_estimator_
best_tree.fit(X_train, y_train)

y_pred = best_tree.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print("\nTest accuracy:", test_acc)
print("\nClassification report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 7920 candidates, totalling 39600 fits
Best CV accuracy: 0.86175
Best params: {'max_depth': None, 'max_leaf_nodes': 26, 'min_samples_leaf': 20}

Test accuracy: 0.8625

Classification report:
               precision    recall  f1-score   support

           0       0.87      0.85      0.86      1000
           1       0.85      0.88      0.86      1000

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000



8. Grow a forest.

a. Continuing the previous exercise, generate 1,000 subsets of the training set,
each containing 100 instances selected randomly. Hint: you can use ScikitLearn’s ShuffleSplit class for this.

b. Train one Decision Tree on each subset, using the best hyperparameter values
found above. Evaluate these 1,000 Decision Trees on the test set. Since they
were trained on smaller sets, these Decision Trees will likely perform worse
than the first Decision Tree, achieving only about 80% accuracy.

c. Now comes the magic. For each test set instance, generate the predictions of the 1,000 Decision Trees, and keep only the most frequent prediction (you can use SciPy’s mode() function for this). This gives you majority-vote predictions over the test set.

d. Evaluate these predictions on the test set: you should obtain a slightly higher
accuracy than your first model (about 0.5 to 1.5% higher). Congratulations,
you have trained a Random Forest classifier!

In [4]:
# Chapter 6, Exercise 8 — “Grow a forest” (manual Random Forest via bagging + majority vote)
# Prereq: run Exercise 7 first to get X_train, X_test, y_train, y_test and best hyperparams.

import numpy as np
from sklearn.model_selection import ShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# If you are continuing directly from Exercise 7, you already have:
# X_train, X_test, y_train, y_test
# and grid.best_params_ / best_tree
# We'll use the best params found in Exercise 7:
best_params = grid.best_params_.copy()  # from your GridSearchCV in Exercise 7

RANDOM_STATE = 42
n_trees = 1000
subset_size = 100

# a) Generate 1,000 random subsets (bootstrap-like, but using ShuffleSplit)
# ShuffleSplit samples WITHOUT replacement by default within each split.
# That's ok for this exercise (the book suggests random subsets).
rs = ShuffleSplit(
    n_splits=n_trees,
    train_size=subset_size,
    random_state=RANDOM_STATE
)

# We'll store predictions of all trees on the test set:
# shape = (n_trees, len(X_test))
all_tree_preds = np.empty((n_trees, len(X_test)), dtype=np.int64)

# b) Train one tree per subset, evaluate individual trees on test set
tree_accs = []

for i, (subset_idx, _) in enumerate(rs.split(X_train)):
    X_subset = X_train[subset_idx]
    y_subset = y_train[subset_idx]

    tree = DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params)
    tree.fit(X_subset, y_subset)

    preds = tree.predict(X_test)
    all_tree_preds[i] = preds

    tree_accs.append(accuracy_score(y_test, preds))

print(f"Individual trees: mean acc = {np.mean(tree_accs):.4f}, "
      f"std = {np.std(tree_accs):.4f}, "
      f"min = {np.min(tree_accs):.4f}, max = {np.max(tree_accs):.4f}")

# c) Majority vote over the 1000 trees (mode along axis=0)
# Use numpy (no SciPy needed): for binary classes, mean>=0.5 works; for general, use bincount.
def majority_vote(pred_matrix: np.ndarray) -> np.ndarray:
    """
    pred_matrix: shape (n_models, n_samples), integer class labels
    returns: shape (n_samples,)
    """
    n_models, n_samples = pred_matrix.shape
    final = np.empty(n_samples, dtype=pred_matrix.dtype)
    for j in range(n_samples):
        votes = pred_matrix[:, j]
        final[j] = np.bincount(votes).argmax()
    return final

y_majority = majority_vote(all_tree_preds)

# d) Evaluate majority vote accuracy
forest_acc = accuracy_score(y_test, y_majority)
print(f"\nMajority-vote 'forest' accuracy = {forest_acc:.4f}")

# Compare to your single best tree from Exercise 7:
single_tree_acc = accuracy_score(y_test, best_tree.predict(X_test))
print(f"Single best tree accuracy       = {single_tree_acc:.4f}")
print(f"Improvement                     = {(forest_acc - single_tree_acc)*100:.2f}%")

Individual trees: mean acc = 0.7689, std = 0.0252, min = 0.6045, max = 0.8295

Majority-vote 'forest' accuracy = 0.7880
Single best tree accuracy       = 0.8625
Improvement                     = -7.45%
